**Author**: Felipe Matheus
**Pipeline**: Cold drawing — **one-step-ahead vs multistep rollout** evaluation

Three complementary views of the SAME trained surrogate (spec §2.3, §4.3):

1. **OOF (training)** — honest generalisation with **group folds** (whole wires
   per fold): the calibration/metrics stored in the bundle's artifacts.
2. **One-step-ahead (validation)** — every pass predicted from TRUE inputs.
   Measures the per-pass model quality with no compounding.
3. **Multistep rollout (validation)** — pass 1 starts from true state; from
   pass 2 on, the state feature (`initial_tensile_strength`) carries the
   model's own previous prediction. Two flavours:
   - **mean-chained** (`Evaluation.rollout_all_groups`): feeds μ̂ forward —
     the deterministic-style rollout;
   - **Monte Carlo** (`ColdDrawingRollout.rollout_all_groups_mc`): K
     trajectories, each pass SAMPLED from the calibrated truncated
     predictive law — the honest, compounded predictive distribution
     (spec §4.3).

The gap between (2) and (3) isolates error **compounding**; the MC σ-growth
curve shows how fast honest uncertainty accumulates per pass. Expect rollout
coverage to degrade vs nominal — that is information, not failure.


# 1. Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.feature_engineering.CDHelper import CDHelper
from src.modeling.Modeling import Modeling
from src.modeling.MBCInference import MBCInference
from src.modeling.CDRollout import ColdDrawingRollout
from src.modeling.CDGraph import ColdDrawingMBCHelper
from src.metrics.Evaluation import Evaluation
from src.metrics.ProbabilisticEvaluation import ProbabilisticEvaluation
from src.DataLoader import LoaderHelper
from config.Variables import Variables

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
cdh  = CDHelper()
modl = Modeling()
evla = Evaluation()
load = LoaderHelper()
mbc  = MBCInference(modl, load)
roll = ColdDrawingRollout(modl)
pev  = ProbabilisticEvaluation()


# 2. Configuration

In [ ]:
# ---- Surrogate bundle (best run auto-picked from the TAG folder) ----
UTS_TAG_DIR = os.path.join(varv.PATHS.models, "cold_drawing_uts",
                           "experiments", "cd-uts-v1-best_quality")
bundles = mbc.load_surrogates_best({"uts": UTS_TAG_DIR}, metric="rmse")
uts_bundle = bundles["uts"]
print(f"target={uts_bundle.target} | features={uts_bundle.features}")

TARGET = uts_bundle.target
SHORT = uts_bundle.short_name           # "tensile_strength"
GROUP_COL = "group_experiment_id"
FILE_NAME = "dataset_cold-drawing_tensile-strength_simulations_12032026.csv"
K_SAMPLES = 200
SEED = 0
ALPHAS = (0.5, 0.8, 0.9, 0.95)


# 3. Data + validation set

In [ ]:
df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
# Glossary-aligned per-pass preparation:
# 1) drop the spreadsheet SPACER rows (blank lines between wires export as
#    all-NaN rows — the real CSV has them);
# 2) simulations only: keep rows whose sanity_check_ok is TRUE;
# 3) cast to float, ignoring identifier/string columns;
# 4) (re)compute total_strain / reduction_ratio from the MEASURED diameters
#    (essays do not store them; single formula source in src.utils);
# 5) build the wire grouping from the previous_experiment_id linkage.
df_raw = df_raw.dropna(subset=["experiment_id"]).reset_index(drop=True)
# spreadsheet placeholder markers ('-') -> NA (e.g. unmeasured purity on a
# whole wire); the feature-wise dropna below then excludes those rows
# honestly. Roots are detected by pass_number, so '-' in
# previous_experiment_id is irrelevant for the grouping.
df_raw = df_raw.replace({"-": pd.NA})
if "sanity_check_ok" in df_raw.columns:
    ok = df_raw["sanity_check_ok"].astype(str).str.upper().isin(["TRUE", "1"])
    df_raw = df_raw[ok].reset_index(drop=True)

df_float = proc.df_to_float(
    df_raw,
    drop_cols=[c for c in ("DOI",) if c in df_raw.columns],
    ignore_columns=[c for c in ("material", "experiment_id",
                                "previous_experiment_id",
                                "sanity_check_ok", "coolant") if c in df_raw.columns],
)
df_geom = cdh.add_derived_geometry(df_float)
df_grouped = cdh.set_group_experiment(df_geom)

df_state = cdh.add_initial_state_column(
    df_grouped,
    final_col=TARGET,
    original_col="original_tensile_strength",
    output_col="initial_tensile_strength",
    group_col=GROUP_COL,
)
cols = [GROUP_COL] + list(uts_bundle.features) + [TARGET]
df = df_state[cols].dropna().reset_index(drop=True)
print(f"Dataset: {df.shape} | wires: {df[GROUP_COL].nunique()}")


In [ ]:
# =====================================================================
# PLACEHOLDER VALIDATION SET — STRUCTURAL ONLY
# df_val = df.copy() means every metric below is IN-SAMPLE and optimistic.
# Replace with real held-out wires (same columns, incl. GROUP_COL) as soon
# as they exist; nothing else in this notebook changes.
# =====================================================================
df_val = df.copy()
print(f"df_val: {df_val.shape} (IN-SAMPLE placeholder — replace with held-out wires)")


# 4. Way 1 — OOF training diagnostics (group folds)

Straight from the bundle's artifacts: these were computed on out-of-fold
predictions with WHOLE WIRES per fold, so they are the honest training-side
reference the two validation views compare against.

In [ ]:
art = uts_bundle.artifacts
print("OOF metrics:", {k: round(v, 3) for k, v in art.get("metrics", {}).items()
                       if isinstance(v, (int, float))})
print("calibration (after scalar c):")
display(pd.DataFrame(art.get("calibration_after", art.get("calibration", []))))
print(f"recalibration_c = {art['recalibration_c']:.4f}")


# 5. Way 2 — One-step-ahead on the validation set

Every pass predicted from TRUE inputs (including the true
`initial_tensile_strength`). This is per-pass model quality with zero
compounding — the best the surrogate can do.

In [ ]:
df_onestep = mbc.predict_surrogate(uts_bundle, df_val)
one_step = evla.one_step_metrics(
    y_true=df_onestep[TARGET].to_numpy(),
    mu=df_onestep[f"{SHORT}_mu"].to_numpy(),
    sigma=df_onestep[f"{SHORT}_sigma"].to_numpy(),
    alphas=ALPHAS,
)
print("One-step metrics:", {k: (round(v,3) if isinstance(v,(int,float)) else v)
                            for k, v in one_step.items() if k != "coverage"})
print("coverage:", {a: round(c,3) for a, c in one_step["coverage"].items()})


In [ ]:
# Per-pass one-step table: intrinsic difficulty per pass with TRUE inputs.
onestep_by_pass = evla.one_step_metrics_by(
    df_onestep, y_col=TARGET, mu_col=f"{SHORT}_mu",
    sigma_col=f"{SHORT}_sigma", by="pass_number", alphas=(0.9,),
)
onestep_by_pass


# 6. Way 3a — Mean-chained rollout (deterministic-style)

`initial_tensile_strength` at pass i>1 is overwritten by the PREVIOUS pass's
predicted mean. Fast, but silently drops the per-pass variance — kept as the
baseline the MC version improves on.

In [ ]:
predict_fn = lambda X: roll.predict_bundle(uts_bundle, X)   # mu / sigma_total

rollout_mean = evla.rollout_all_groups(
    df_val, group_col=GROUP_COL, predict_fn=predict_fn,
    features=list(uts_bundle.features), target=TARGET,
    recursive_feature="initial_tensile_strength",
)
final_mean = evla.final_state_metrics(rollout_mean, alphas=ALPHAS)
print("Mean-chained FINAL-state metrics:",
      {k: (round(v,3) if isinstance(v,(int,float)) else v)
       for k, v in final_mean.items() if k != "coverage"})
print("final coverage:", {a: round(c,3) for a, c in final_mean["coverage"].items()})


# 6.1 Way 3b — Monte Carlo rollout (spec §4.3)

K trajectories per wire; every pass SAMPLES the truncated predictive law and
the SAMPLES feed the next pass. `sigma_pred` per pass is the empirical spread
of the K trajectories — the honest compounded uncertainty.

In [ ]:
rollout_mc = roll.rollout_all_groups_mc(
    bundles, df_val, group_col=GROUP_COL,
    k_samples=K_SAMPLES, seed=SEED,
)
rollout_mc_uts = rollout_mc[rollout_mc.surrogate == SHORT].reset_index(drop=True)
final_mc = evla.final_state_metrics(rollout_mc_uts, alphas=ALPHAS)
print("MC-rollout FINAL-state metrics:",
      {k: (round(v,3) if isinstance(v,(int,float)) else v)
       for k, v in final_mc.items() if k != "coverage"})
print("final coverage:", {a: round(c,3) for a, c in final_mc["coverage"].items()})


# 7. Pass-by-pass curves — where does compounding bite?

In [ ]:
curve_mean = evla.pass_by_pass_curve(rollout_mean)
curve_mc   = evla.pass_by_pass_curve(rollout_mc_uts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.plot(onestep_by_pass["pass_number"], onestep_by_pass["rmse"],
        "o-", label="one-step RMSE (true inputs)")
ax.plot(curve_mean["pass_index"], curve_mean["mean_abs_error"],
        "s--", label="mean-chained rollout |err|")
ax.plot(curve_mc["pass_index"], curve_mc["mean_abs_error"],
        "d-", label="MC rollout |err|")
ax.set_xlabel("pass"); ax.set_ylabel("error (MPa)")
ax.set_title("Error per pass: compounding gap"); ax.legend(); ax.grid(alpha=.3)

ax = axes[1]
ax.plot(curve_mc["pass_index"], curve_mc["mean_sigma"],
        "d-", label="MC rollout mean σ (compounded)")
ax.plot(curve_mean["pass_index"], curve_mean["mean_sigma"],
        "s--", label="mean-chained mean σ (per-pass only)")
ax.set_xlabel("pass"); ax.set_ylabel("σ (MPa)")
ax.set_title("Honest σ growth per pass"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Coverage per pass: nominal 90% vs empirical under rollout. Degradation
# with pass index is EXPECTED (the one-step calibration does not promise
# multi-step coverage); it quantifies how far the promise stretches.
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(curve_mc["pass_index"], curve_mc["coverage_90"], "d-", label="MC rollout")
ax.plot(curve_mean["pass_index"], curve_mean["coverage_90"], "s--", label="mean-chained")
ax.axhline(0.9, color="k", lw=1, ls=":", label="nominal 0.90")
ax.set_xlabel("pass"); ax.set_ylabel("coverage @ α=0.9")
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Interval coverage under rollout")
plt.show()


# 8. Compounding ratio — one number for the story

In [ ]:
n_typ = int(df_val.groupby(GROUP_COL)["pass_number"].max().median())
ratio_mean = evla.compounding_ratio(final_mean, one_step, n_passes_typical=n_typ)
ratio_mc   = evla.compounding_ratio(final_mc,   one_step, n_passes_typical=n_typ)
print(f"typical passes per wire: {n_typ}")
print(f"compounding ratio (mean-chained): {ratio_mean:.2f}")
print(f"compounding ratio (MC):           {ratio_mc:.2f}")
print("~1.0 = errors compound like independent noise (sqrt(n) growth);")
print(">1.0 = correlated errors compounding faster — the honest penalty of chaining.")


# 9. Reading the results

- **one-step vs rollout gap** = pure compounding: if the rollout final-state
  RMSE is much larger than one-step, the wire-level story (what the MBC will
  face) is harder than the per-pass metrics suggest.
- **MC σ growth** should track the rollout |error| growth — when it does, the
  compounded uncertainty is HONEST, and the MBC can trust `Pr(success)` per
  route. When σ grows slower than the error, the chain is overconfident:
  revisit calibration or the aleatoric model.
- These numbers are on the in-sample placeholder; re-run with the held-out
  wires before drawing conclusions.